# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ADHIRAJ994/Fly-Rank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane: Lane 2 — Refresh / Content Opportunity Scoring.** This notebook takes that lane from the starter CSV (w01/w02) onto the real warehouse (`fact_content_daily_performance`), for a single mid-panel month (`month=2026-03`).

## 0. Setup — HF token, DuckDB, and finding the right files

*Not part of the contract itself — just wiring so every query cell below actually runs. We use DuckDB to query the parquet files directly on Hugging Face without downloading the full 79M-row table.*

In [ ]:
# Colab: HF_TOKEN must already be saved as a Secret (key icon, left sidebar) — never pasted here, repo is public.
import os, duckdb

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    assert os.environ.get('HF_TOKEN'), 'Set HF_TOKEN as a Colab Secret or env var before continuing.'

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("""
CREATE OR REPLACE SECRET hf_token (
    TYPE huggingface,
    TOKEN getenv('HF_TOKEN')
);
""")
print('DuckDB + HF secret ready.')

In [ ]:
# Discover the exact parquet path pattern for the month=2026-03 partition before guessing at globs.
# (Repo layout can differ slightly from the README's row-count table — always check the real listing once per session.)
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files('FlyRank/internship-warehouse', repo_type='dataset')

daily_files = [f for f in files if 'fact_content_daily_performance' in f and 'sample' not in f]
march_files = [f for f in daily_files if 'month=2026-03' in f]

print(f"Total fact_content_daily_performance files (non-sample): {len(daily_files)}")
print(f"Files under month=2026-03: {len(march_files)}")
print()
print('Sample paths:')
for f in march_files[:5]:
    print(' ', f)

# Adjust this glob if the printed paths above look different from what's assumed here.
MARCH_GLOB = f"hf://datasets/FlyRank/internship-warehouse/{march_files[0].rsplit('/', 1)[0]}/*.parquet" if march_files else None
print()
print('MARCH_GLOB =', MARCH_GLOB)

*If `march_files` comes back empty, the partition folder is named differently than assumed — print `daily_files[:20]` to see the real layout and hand-fix `MARCH_GLOB` before moving on. Do this once, then reuse `MARCH_GLOB` for every query below so you are not re-globbing the whole 79M-row table each time.*

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row (source grain) = one `(client_id, content_id, report_date)` daily performance record** — one content item's search + engagement metrics for one calendar day, from `fact_content_daily_performance`.

**For this lane, I re-grain to one row = one `(client_id, content_id)` monthly snapshot**, built by aggregating the daily rows within `month=2026-03` into two halves:
- **decision moment = end of day 2026-03-15** — everything I'm allowed to use as a feature is aggregated from `report_date` 2026-03-01 through 2026-03-15.
- **outcome window = 2026-03-16 through 2026-03-31** — this is where the decline label/proxy is measured from. Nothing in this window is a feature.

**Time window:** `month=2026-03` only (a mid-panel month, per the warehouse skill's iteration rule). The sealed `_sample` table (June 2026, the panel's final month) is never touched here — it is the natural outcome window of any real past→future label and is reserved for query-mechanics testing only.

**What I'd predict/rank (label or proxy):** *Is this content item declining?* — proxied here as: did total clicks in the outcome window (days 16–31) fall below total clicks in the decision window (days 1–15)? This mirrors the starter CSV's `trend_direction`, but built fresh from warehouse daily rows instead of trusting a pre-computed column.

**One thing deliberately excluded:** any `fact_content_query_90d` columns. That table's 90-day window overlaps the snapshot's final months, and per the warehouse skill, only `*_prev30`-style columns from it would be window-safe — I'm not pulling from it at all this week to keep the contract to one clean source table.

In [ ]:
# Text answer is in the markdown cell above. This cell is left as a placeholder per the skeleton's rule
# (text lives above, code lives here) — the actual verification runs in Section 3.
pass

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `gsc_avg_position` (avg, days 1–15), `impressions` (sum, days 1–15), `clicks` (sum, days 1–15), `ctr_h1` (derived: clicks_h1/impressions_h1), `engaged_sessions` (sum, days 1–15, gated by availability) | All aggregated only from `report_date` 2026-03-01–2026-03-15 — knowable at the decision moment, before the outcome window exists. |
| **Label / proxy** | `clicks` (sum, days 16–31) → `decline_label` = 1 if clicks_h2 < clicks_h1 else 0 | This is what I'm predicting. `clicks_h2` and anything derived from it never becomes a feature. |
| **Context** | `client_id`, `content_id`, `report_date`, `month`, `ga4_data_available` | Grouping, joining, filtering, and windowing only — never fed to a model as signal. |
| **Excluded** | `fact_content_query_90d.*` (window overlaps outcome period — unsafe without per-column alignment work I'm not doing this week); rows where `ga4_data_available IS NOT TRUE` for any GA4-sourced feature (zero-filled, not truly zero — would inject a false 'no engagement' signal); any content item with fewer than 5 days of impressions in *either* half (proxy label too noisy on near-zero-traffic items) | Each excluded for a specific measurement-validity reason, not convenience. |

In [ ]:
# Text answer is in the markdown cell above.
pass

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3a. Grain check — one row really is what I said

In [ ]:
# Grain claim: (client_id, content_id, report_date) is unique in the March slice of fact_content_daily_performance.
grain_check = con.sql(f"""
SELECT client_id, content_id, report_date, COUNT(*) AS c
FROM '{MARCH_GLOB}'
GROUP BY client_id, content_id, report_date
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print(f"Duplicate grain rows found: {len(grain_check)}")
grain_check

*Expect an empty result above — zero rows back means the grain holds. If it isn't empty, the unit of analysis in Section 1 is wrong, not the query.*

### 3b. Slice size + date span

In [ ]:
slice_stats = con.sql(f"""
SELECT
    COUNT(*)                     AS total_rows,
    COUNT(DISTINCT client_id)    AS n_clients,
    COUNT(DISTINCT content_id)   AS n_content_items,
    MIN(report_date)             AS earliest_date,
    MAX(report_date)             AS latest_date
FROM '{MARCH_GLOB}'
""").df()

slice_stats

*Sentence this backs up: `month=2026-03` contains N daily rows spanning 2026-03-01 to 2026-03-31 across the printed client/content counts — fill in the printed numbers here once run.*

### 3c. Availability — `IS TRUE` filter, rows survived

In [ ]:
availability = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_available
FROM '{MARCH_GLOB}'
""").df()

availability

*This is the check the warehouse skill warns about directly: rows before a client's `ga4_data_start` are zero-filled with the flag FALSE, not genuinely zero-engagement. Any GA4-sourced feature below only uses rows where this is TRUE.*

### 3d. Build the feature frame (five features, decision window = days 1–15)

Every feature below: **knowable at the decision moment because** it is aggregated only from `report_date` 2026-03-01 through 2026-03-15 — nothing from the outcome window (16–31) touches this frame.

In [ ]:
feature_frame = con.sql(f"""
WITH h1 AS (
    SELECT
        client_id,
        content_id,
        AVG(gsc_avg_position) AS avg_position_h1,
        SUM(impressions)      AS impressions_h1,
        SUM(clicks)           AS clicks_h1,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN engaged_sessions ELSE NULL END) AS engaged_sessions_h1,
        COUNT(*) FILTER (WHERE impressions > 0) AS days_with_impressions_h1
    FROM '{MARCH_GLOB}'
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY client_id, content_id
)
SELECT
    client_id,
    content_id,
    avg_position_h1,
    impressions_h1,
    clicks_h1,
    ROUND(clicks_h1::DOUBLE / NULLIF(impressions_h1, 0), 4) AS ctr_h1,
    engaged_sessions_h1,
    days_with_impressions_h1
FROM h1
WHERE days_with_impressions_h1 >= 5  -- excludes near-zero-traffic items, per Section 2
""").df()

print(f"Feature frame rows: {len(feature_frame):,}")
feature_frame.head()

**Five features and why each is knowable at the decision moment:**

1. `avg_position_h1` — average `gsc_avg_position` over days 1–15. Search Console reports this daily; nothing here reaches past day 15.
2. `impressions_h1` — summed impressions, days 1–15. Same reasoning — already-observed GSC data.
3. `clicks_h1` — summed clicks, days 1–15. Already-observed GSC data, decision-window only.
4. `ctr_h1` — `clicks_h1 / impressions_h1`. Purely derived from the two features above; no new information from outside the window.
5. `engaged_sessions_h1` — summed engaged sessions, days 1–15, **only** for rows where `ga4_data_available IS TRUE`. Gating on the flag (per 3c) means this is real GA4 signal already observed by day 15, not a zero-fill artifact.

### 3e. The label (outcome window, days 16–31) — built separately, never joined into features above

In [ ]:
label_frame = con.sql(f"""
SELECT
    client_id,
    content_id,
    SUM(clicks) AS clicks_h2
FROM '{MARCH_GLOB}'
WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
GROUP BY client_id, content_id
""").df()

dataset = feature_frame.merge(label_frame, on=['client_id', 'content_id'], how='inner')
dataset['decline_label'] = (dataset['clicks_h2'] < dataset['clicks_h1']).astype(int)

print(f"Rows with both halves present: {len(dataset):,}")
print(f"Decline rate (proxy label): {dataset['decline_label'].mean():.1%}")
dataset.head()

### 3f. The trap — deliberate leakage, then removed

On purpose: add `clicks_h2` (the label's own input) as if it were a feature, train a quick model, watch the score jump toward perfect. Then delete it and keep the honest number — this is the notebook-02 leakage lesson, done here on real warehouse data.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ['avg_position_h1', 'impressions_h1', 'clicks_h1', 'ctr_h1', 'engaged_sessions_h1']
X = dataset[honest_features].fillna(0)
y = dataset['decline_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

model_honest = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, model_honest.predict_proba(X_test)[:, 1])
print(f"HONEST score (features from days 1-15 only), AUC: {honest_auc:.3f}")

In [ ]:
# --- THE TRAP: add clicks_h2 (the label's own ingredient) as a 'feature' ---
leaky_features = honest_features + ['clicks_h2']
X_leak = dataset[leaky_features].fillna(0)

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leak, y, test_size=0.25, random_state=42, stratify=y)

model_leaky = LogisticRegression(max_iter=1000).fit(X_train_l, y_train_l)
leaky_auc = roc_auc_score(y_test_l, model_leaky.predict_proba(X_test_l)[:, 1])

print(f"LEAKY score (clicks_h2 included as a 'feature'), AUC: {leaky_auc:.3f}")
print(f"Jump from honest to leaky: +{leaky_auc - honest_auc:.3f}")
print()
print("clicks_h2 is literally half the definition of decline_label (clicks_h2 < clicks_h1) —")
print("the model isn't learning a pattern, it's reading the answer key.")

In [ ]:
# --- Remove the leak, keep the honest number ---
del model_leaky, X_leak, X_train_l, X_test_l, y_train_l, y_test_l, leaky_features

print(f"FINAL, KEPT score for this lane at month=2026-03: AUC = {honest_auc:.3f} (honest, days 1-15 features only)")
print(f"Discarded: the {leaky_auc:.3f}-AUC run that used clicks_h2 — that number does not count.")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation:** history depth and GA4 coverage differ wildly per client (per `dim_clients.gsc_data_start` / GA4 availability flag checked in 3c). For clients whose GA4 tracking started partway through, or after, 2026-03-01, `engaged_sessions_h1` is silently `NULL`/excluded for their content — not because those pages have no engagement, but because the warehouse has no record for that window yet. That means the feature frame under-represents newer or later-onboarded clients relative to long-tenured ones, and the honest AUC above is measured on a panel that is not evenly balanced across clients. This data can never tell me whether a content item with no GA4 rows in this window is actually low-engagement or just unmeasured — those two cases are indistinguishable from `NULL` alone.

In [ ]:
# Show the imbalance named above, per-client, for this month's slice.
client_coverage = con.sql(f"""
SELECT
    client_id,
    COUNT(*) AS rows_total,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_ga4_available,
    ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_available
FROM '{MARCH_GLOB}'
GROUP BY client_id
ORDER BY pct_available ASC
LIMIT 10
""").df()

print("Ten clients with the LOWEST GA4 availability this month (the ones the limitation above is about):")
client_coverage

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

> Run this top to bottom in Colab first — `MARCH_GLOB` in Section 0 depends on the real HF file listing, and every number quoted in the markdown cells above (row counts, AUC values, decline rate) needs to match what your own run actually prints before you commit.